In [1]:
import pandas as pd
df = pd.read_csv("NGCC_Uncertainty_dataset.csv")

In [2]:
df

,Ambient Air Temperature (Dry Bulb Average) (°F),Natural Gas Cost ($/mscf),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg),Important Performance and Cost Factors: CO2 Removed (tons/hr),Important Performance and Cost Factors: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton),Makeup Water (tons/hr)
0,75.137497,6.725,33.900002,60.674999,4.4425,135.200363,491.935730,328.892700,0.000000
1,69.612503,5.875,43.700001,81.525002,5.9275,182.987564,477.473724,228.331604,0.000000
2,84.912498,11.325,33.299999,61.875000,1.6525,136.096603,499.090912,449.124939,0.000000
3,54.737499,2.425,15.300000,82.724998,5.5225,189.321732,489.426849,159.739182,789.642822
4,18.187500,8.875,36.099998,63.375000,1.7425,151.950363,547.840698,362.821564,625.265442
...,...,...,...,...,...,...,...,...,...
195,75.987503,9.025,11.700000,68.025002,2.8225,151.407898,496.516418,361.050995,882.480652
196,59.837502,9.875,32.700001,81.224998,6.3325,184.661026,482.363220,311.367065,199.811310
197,97.237503,9.725,19.100000,64.724998,5.4325,140.030426,469.309143,394.297119,0.000000
198,57.287498,3.975,12.700000,62.174999,9.3475,141.821533,477.319916,253.032211,772.738037


In [3]:
df.isna().sum()

Ambient Air Temperature (Dry Bulb Average) (°F)                       0
Natural Gas Cost ($/mscf)                                             0
Cooling Water Temperature Rise (°F)                                   0
Absorber CO2 Removal Efficiency (%)                                   0
Amine Scrubber Power Requirement (% MWg)                              0
Important Performance and Cost Factors: CO2 Removed (tons/hr)         0
Important Performance and Cost Factors: Net Electrical Output (MW)    0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)         0
Makeup Water (tons/hr)                                                0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Ambient Air Temperature (Dry Bulb Average) (°F)',
    'Natural Gas Cost ($/mscf)',
    'Cooling Water Temperature Rise (°F)',
    'Absorber CO2 Removal Efficiency (%)',
    'Amine Scrubber Power Requirement (% MWg)']]

y = df['Important Performance and Cost Factors: CO2 Removed (tons/hr)']

In [6]:
x

,Ambient Air Temperature (Dry Bulb Average) (°F),Natural Gas Cost ($/mscf),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg)
0,75.137497,6.725,33.900002,60.674999,4.4425
1,69.612503,5.875,43.700001,81.525002,5.9275
2,84.912498,11.325,33.299999,61.875000,1.6525
3,54.737499,2.425,15.300000,82.724998,5.5225
4,18.187500,8.875,36.099998,63.375000,1.7425
...,...,...,...,...,...
195,75.987503,9.025,11.700000,68.025002,2.8225
196,59.837502,9.875,32.700001,81.224998,6.3325
197,97.237503,9.725,19.100000,64.724998,5.4325
198,57.287498,3.975,12.700000,62.174999,9.3475


In [7]:
y

0      135.200363
1      182.987564
2      136.096603
3      189.321732
4      151.950363
          ...    
195    151.407898
196    184.661026
197    140.030426
198    141.821533
199    168.934784
Name: Important Performance and Cost Factors: CO2 Removed (tons/hr), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== PYOMO MODEL ===\n")


=== PYOMO MODEL ===



In [9]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 2, 'reg__alpha': 0.001}
Train R2: 0.9999999698895434
Test R2: 0.9999999475041553


In [10]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Polynomial | RMSE: 0.0046 | MAPE: 0.0020%


In [11]:
import numpy as np, json, joblib

joblib.dump(best_poly, "NGCC_CO2_Removed_poly.pkl")

poly   = best_poly.named_steps['poly']
scaler = best_poly.named_steps['scaler']
reg    = best_poly.named_steps['reg']

coef_real      = reg.coef_ / scaler.scale_
intercept_real = reg.intercept_ - np.sum(reg.coef_ * scaler.mean_ / scaler.scale_)

feature_names = list(x.columns)
powers = poly.powers_
terms = [{"coef": float(c),
          "powers": {feature_names[j]: int(p[j]) for j in range(len(feature_names)) if p[j] > 0}}
         for p, c in zip(powers, coef_real)]
equation = {"intercept": float(intercept_real), "degree": int(poly.degree), "terms": terms}

P = poly.transform(x_test)
assert np.allclose(intercept_real + P @ coef_real, best_poly.predict(x_test)), "mismatch!"

json.dump(equation, open("NGCC_CO2_Removed_poly_eq.json", "w"), indent=2)
print("saved. degree =", equation["degree"], "| terms =", len(terms))

saved. degree = 2 | terms = 20
